In [ ]:
"""Build the training dataset for the player recommender model.

This script performs feature engineering for the player recommender. It loads
the scraped source data, creates season-based player features, builds the final
supervised training dataset, and saves it as a CSV file.
"""

import sys
from pathlib import Path

import pandas as pd

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from ml.toolkit.ml_utilities import (
    build_recommender_season_features,
    build_recommender_training_dataset,
)


SOURCE_DATA_DIR = PROJECT_ROOT / "data" / "scrape" / "amateur"
OUTPUT_PATH = CURRENT_DIR / "recommender_model_dataset.csv"

FIRST_TRAINING_SEASON = 2020
LAST_TRAINING_SEASON = 2024


def load_source_tables(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the source tables required for recommender feature engineering.

    The function checks whether all required CSV files are available and then
    loads player statistics, match information, and player information from the
    source data directory.
    """
    required_files = {
        "player_stats": data_dir / "player_stats.csv",
        "matches": data_dir / "matches.csv",
        "players": data_dir / "players.csv",
    }

    missing_files = [str(path) for path in required_files.values() if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing source files: {missing_files}")

    player_stats = pd.read_csv(required_files["player_stats"])
    matches = pd.read_csv(required_files["matches"])
    players = pd.read_csv(required_files["players"])

    return player_stats, matches, players


def build_dataset(
    player_stats: pd.DataFrame,
    matches: pd.DataFrame,
    players: pd.DataFrame,
) -> pd.DataFrame:
    """Create the supervised recommender dataset from raw source tables.

    The function first builds season-level player features from the available
    source data. These features are then transformed into a training dataset for
    the recommender model within the configured season range.
    """
    season_features = build_recommender_season_features(
        player_stats,
        matches,
        players,
    )

    dataset = build_recommender_training_dataset(
        season_features,
        first_training_season=FIRST_TRAINING_SEASON,
        last_training_season=LAST_TRAINING_SEASON,
    )

    return dataset


def save_dataset(dataset: pd.DataFrame, output_path: Path) -> None:
    """Save the recommender training dataset to disk.

    The function creates the output directory if necessary, writes the dataset as
    a CSV file, and prints a short summary with the number of rows, unique
    players, and available seasons.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    dataset.to_csv(output_path, index=False)

    print(f"Saved recommender dataset: {output_path}")
    print(f"Rows: {len(dataset)}")
    print(f"Players: {dataset['player_id'].nunique()}")
    print(dataset["season"].value_counts().sort_index())


def main() -> None:
    """Run the complete recommender feature engineering pipeline.

    The function loads the source tables, builds the model-ready recommender
    dataset, and saves the resulting CSV file to the configured output path.
    """
    player_stats, matches, players = load_source_tables(SOURCE_DATA_DIR)
    dataset = build_dataset(player_stats, matches, players)
    save_dataset(dataset, OUTPUT_PATH)


if __name__ == "__main__":
    main()